In [ ]:
# FINAL FIXED TRAINING — FULL FINE-TUNING, CORRECT PIPELINE, CORRECT LOSS
# -----------------------------------------------------------------------

!pip install -q kagglehub --upgrade
import kagglehub, os, shutil, tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# ======================================================
# Load + Clean dataset
# ======================================================
root_path = kagglehub.dataset_download("salmansajid05/oral-diseases")
src_path = "/kaggle/input/oral-diseases"
clean_path = "/content/oral_clean"

allowed = ["Calculus","Data caries","Gingivitis","Mouth Ulcer","Tooth Discoloration","hypodontia"]

os.makedirs(clean_path, exist_ok=True)
for cls in allowed:
    shutil.copytree(os.path.join(src_path, cls),
                    os.path.join(clean_path, cls),
                    dirs_exist_ok=True)

print("Classes:", os.listdir(clean_path))

# ======================================================
# TF.DATA LOADING
# ======================================================
IMG_SIZE = 300
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    clean_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    clean_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Detected classes:", class_names)

# Prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# ======================================================
# Class Weights
# ======================================================
all_labels = [y.numpy() for _, y in train_ds.unbatch()]
class_weights = dict(enumerate(compute_class_weight(
    class_weight="balanced",
    classes=np.unique(all_labels),
    y=all_labels
)))
print("Class weights:", class_weights)

# ======================================================
# EfficientNetB3 — Proper Classification Head
# ======================================================

base = tf.keras.applications.EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base.trainable = False  # Phase 1

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)

x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print(model.summary())

# ======================================================
# PHASE 1 — Train top layers ONLY (quick warmup)
# ======================================================
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weights
)

# ======================================================
# PHASE 2 — TRUE FINE-TUNING (UNFREEZE LAST ~120 LAYERS)
# ======================================================
base.trainable = True

for layer in base.layers[:-120]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ======================================================
# Train FULL Fine-tuning
# ======================================================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights
)

print("Training DONE!")

Using Colab cache for faster access to the 'oral-diseases' dataset.
Classes: ['Data caries', 'Gingivitis', 'Tooth Discoloration', 'hypodontia', 'Calculus', 'Mouth Ulcer']
Found 12320 files belonging to 6 classes.
Using 9856 files for training.
Found 12320 files belonging to 6 classes.
Using 2464 files for validation.
Detected classes: ['Calculus', 'Data caries', 'Gingivitis', 'Mouth Ulcer', 'Tooth Discoloration', 'hypodontia']
Class weights: {0: np.float64(1.5810073788899583), 1: np.float64(0.7829679059421671), 2: np.float64(0.8668425681618294), 3: np.float64(0.7349739000745712), 4: np.float64(1.0183922297995454), 5: np.float64(1.6830601092896176)}


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 10, 10, 1536)   │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,178,549 (42.64 MB)

 Trainable params: 395,014 (1.51 MB)

 Non-trainable params: 10,783,535 (41.14 MB)

None
Epoch 1/5
308/308 ━━━━━━━━━━━━━━━━━━━━ 96s 190ms/step - accuracy: 0.6380 - loss: 0.9033 - val_accuracy: 0.8271 - val_loss: 0.4355
Epoch 2/5
308/308 ━━━━━━━━━━━━━━━━━━━━ 48s 154ms/step - accuracy: 0.7999 - loss: 0.4807 - val_accuracy: 0.8498 - val_loss: 0.3620
Epoch 3/5
308/308 ━━━━━━━━━━━━━━━━━━━━ 83s 156ms/step - accuracy: 0.8414 - loss: 0.3863 - val_accuracy: 0.8429 - val_loss: 0.3518
Epoch 4/5
308/308 ━━━━━━━━━━━━━━━━━━━━ 47s 152ms/step - accuracy: 0.8498 - loss: 0.3723 - val_accuracy: 0.8701 - val_loss: 0.3139
Epoch 5/5
308/308 ━━━━━━━━━━━━━━━━━━━━ 47s 153ms/step - accuracy: 0.8740 - loss: 0.3148 - val_accuracy: 0.8762 - val_loss: 0.2951
Epoch 1/15
308/308 ━━━━━━━━━━━━━━━━━━━━ 154s 284ms/step - accuracy: 0.6541 - loss: 0.8815 - val_accuracy: 0.8462 - val_loss: 0.3864
Epoch 2/15
308/308 ━━━━━━━━━━━━━━━━━━━━ 72s 234ms/step - accuracy: 0.8040 - loss: 0.4784 - val_accuracy: 0.8774 - val_loss: 0.3023
Epoch 3/15
308/308 ━━━━━━━━━━━━━━━━━━━━ 71s 232ms/step - accuracy: 0.8492 - loss: 

In [ ]:
import json
from google.colab import files

# ======================================================
# 1. SAVE CLASS INDICES
# ======================================================
class_index_path = "class_indices.json"
with open(class_index_path, "w") as f:
    json.dump({i: name for i, name in enumerate(class_names)}, f, indent=4)

print("Class indices saved to:", class_index_path)

# ======================================================
# 2. SAVE MODEL AS .keras (recommended format)
# ======================================================
model_keras_path = "oral_diseases_model.keras"
model.save(model_keras_path, save_format="keras")
print("Saved:", model_keras_path)

# ======================================================
# 3. SAVE MODEL AS .h5 (legacy format)
# ======================================================
model_h5_path = "oral_diseases_model.h5"
model.save(model_h5_path, save_format="h5")
print("Saved:", model_h5_path)

# ======================================================
# 4. DOWNLOAD ALL FILES
# ======================================================
files.download(model_keras_path)
files.download(model_h5_path)
files.download(class_index_path)

Class indices saved to: class_indices.json


Saved: oral_diseases_model.keras
Saved: oral_diseases_model.h5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>